# 14 · Run bounded concurrent queries

Compare serial and concurrent execution of independent real-data queries while reconciling every result.

Open this notebook in Google Colab with a CPU runtime, or use a Linux x86-64
Jupyter environment with Python 3.11 or newer. No database account is required.
Run the cells in order. The first cell downloads a verified Kelvo preview binary
and installs pinned Python readers; it does not compile Kelvo.

Every analytical SQL query below runs through the **Kelvo executable**. PyArrow
reads its Arrow output, and pandas/matplotlib format small results. The pinned
DuckDB driver materializes execution before delivering Arrow batches; delivery
backpressure is not a worker memory bound.


**Data:** [NYC TLC trip records and taxi-zone lookup](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).
This lesson uses the January 2024 yellow-taxi Parquet file. See the
[NYC Open Data terms of use](https://opendata.cityofnewyork.us/overview/#termsofuse)
and TLC's data dictionary and accuracy notice; do not treat public availability
as a warranty or apply the notebook's code license to the data.
The original Parquet download is roughly 50 MB and contains roughly three million
rows. The helper checks the pinned file digest. Queries use explicit result
limits; a small result does not imply a small source scan.


[All notebooks](https://github.com/SYNEHQ/kelvo-go/blob/cargo/notebooks/README.md)
| [Kelvo documentation](https://github.com/SYNEHQ/kelvo-go/blob/cargo/docs/README.md)

## Set up a private lab


In [ ]:
# Default setup installs pinned Python packages and verifies the reviewed helper.
# Operators validating a checkout can explicitly set KELVO_NOTEBOOK_SKIP_INSTALL=1
# and KELVO_NOTEBOOK_HELPER=/absolute/path/to/kelvo_notebooks.py before starting.
import hashlib
import importlib.util
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import urllib.parse
import urllib.request

KELVO_NOTEBOOK_REF = "eacb7e2aedd8020575ecfede53db819112c40199"
KELVO_HELPER_SHA256 = "86696a66a3c903eb8f18d529fcef055e8bee72234da324b8776529138436d9a9"
if os.environ.get("KELVO_NOTEBOOK_SKIP_INSTALL") != "1":
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet",
        "pyarrow==25.0.1", "pandas==2.3.3", "matplotlib==3.10.7", "PyYAML==6.0.3",
    ], check=True)

helper_directory = None
local_helper = os.environ.get("KELVO_NOTEBOOK_HELPER")
if local_helper:
    helper_path = Path(local_helper).expanduser().resolve(strict=True)
else:
    helper_url = (
        "https://raw.githubusercontent.com/SYNEHQ/kelvo-go/"
        + KELVO_NOTEBOOK_REF + "/notebooks/kelvo_notebooks.py"
    )
    request = urllib.request.Request(helper_url, headers={"User-Agent": "kelvo-notebooks"})
    with urllib.request.urlopen(request, timeout=60) as response:
        final_url = response.geturl()
        if urllib.parse.urlsplit(final_url).scheme != "https" or final_url != helper_url:
            raise ValueError("Helper download did not retain its pinned HTTPS URL")
        payload = response.read(1024 * 1024 + 1)
    if len(payload) > 1024 * 1024:
        raise ValueError("Unexpected helper size")
    if hashlib.sha256(payload).hexdigest() != KELVO_HELPER_SHA256:
        raise ValueError("Helper checksum mismatch")
    helper_directory = Path(tempfile.mkdtemp(prefix="kelvo-notebook-helper-"))
    helper_path = helper_directory / "kelvo_notebooks.py"
    helper_path.write_bytes(payload)

spec = importlib.util.spec_from_file_location("kelvo_notebooks", helper_path)
helper = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helper
spec.loader.exec_module(helper)
from kelvo_notebooks import Lab, remote_query

lab = Lab.create()
print("Kelvo workspace:", lab.root)


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from decimal import Decimal
import time
import pandas as pd
taxi = lab.taxi()
config = lab.catalog(sources=[{"id": "trips", "type": "parquet", "path": str(taxi)}])
JOB_COUNT = 4
MAX_WORKERS = 2
# Larger opt-in exercise: JOB_COUNT=10 and MAX_WORKERS=2 after reviewing VM resources.
assert 1 <= JOB_COUNT <= 10
assert 1 <= MAX_WORKERS <= 2
sql = """
    SELECT COUNT(*)::BIGINT AS trips,
           SUM(total_amount::DECIMAL(18,2)) AS collected_amount
    FROM trips WHERE MOD(COALESCE("PULocationID", 0), ?) = ?
"""
def run_job(index):
    result = lab.query(config, sql, sources=["trips"], max_rows=1, parameters=[
        {"type": "int64", "value": str(JOB_COUNT)},
        {"type": "int64", "value": str(index)},
    ])
    return {"job": index, "row": result.table().to_pylist()[0],
            "seconds": result.elapsed_seconds}


Each job starts its own Kelvo CLI process and has its own query budget.
The Python pool controls how many run together; this is not a cluster test or
single-query distribution. Source scans may overlap and contend for CPU, memory
and cache. Two 512 MiB query budgets are not a 1 GiB process-RSS guarantee.


In [ ]:
started = time.perf_counter()
serial = [run_job(index) for index in range(JOB_COUNT)]
serial_seconds = time.perf_counter() - started
started = time.perf_counter()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    concurrent = list(pool.map(run_job, range(JOB_COUNT)))
concurrent_seconds = time.perf_counter() - started
assert [item["row"] for item in serial] == [item["row"] for item in concurrent]
total = lab.query(config, """
    SELECT COUNT(*)::BIGINT AS trips,
           SUM(total_amount::DECIMAL(18,2)) AS collected_amount FROM trips
""", sources=["trips"]).table().to_pylist()[0]
assert sum(item["row"]["trips"] for item in concurrent) == total["trips"]
assert sum((item["row"]["collected_amount"] or Decimal(0) for item in concurrent), Decimal(0)) == total["collected_amount"]
display(pd.DataFrame([{"job": item["job"], **item["row"], "seconds": item["seconds"]}
                      for item in concurrent]))
print({"serial_wall_seconds": serial_seconds, "concurrent_wall_seconds": concurrent_seconds,
       "workers": MAX_WORKERS, "jobs": JOB_COUNT})


There is deliberately no assertion that concurrency must improve latency.
Serial runs warm the page cache before concurrent runs, and Colab allocation
varies. This workflow tests exact answers under bounded overlap. Repeat in
randomized order on a fixed deployment before making capacity claims.

Kelvo cluster mode adds tenant-bound dispatch, admission and worker lifecycles.
See [cluster operation](https://github.com/SYNEHQ/kelvo-go/blob/cargo/docs/cluster.md)
and [resource admission](https://github.com/SYNEHQ/kelvo-go/blob/cargo/docs/operations.md).


## Clean up

Run this cell after inspecting or saving your results. Re-run the setup and data
cells if you want to restart the lesson after cleanup.


In [ ]:
# Close local processes and delete this lesson's private temporary workspace.
# Downloaded release/data caches, if configured by the helper, remain reusable.
lab.close()
if helper_directory is not None:
    shutil.rmtree(helper_directory, ignore_errors=True)
